# Replicating SCM4OPT 1998-2012 Warming Slowdown with FaIR

This notebook demonstrates how to design the experiment from the Nature Communications paper: *"Reductions in atmospheric levels of non-CO2 greenhouse gases explain about a quarter of the 1998-2012 warming slowdown"* (Su et al., 2024) using **FaIR**, an open-source Simple Climate Model (SCM).

### Methodology Overview
To replicate their findings without their internal SCM4OPT code, we set up two simulations:
1. **Baseline (Historical) Run**: Global temperature simulated using actual historical emissions of all GHGs.
2. **Counterfactual Run (No mitigation in ODS/CH4)**: A simulation where Ozone Depleting Substances (ODS) and Methane (CH4) emissions do *not* decline after the late 1990s, but instead remain fixed at their peak/1998 levels.

The difference in the temperature trend (ΔT/decade) between 1998 and 2012 in these two scenarios isolate the warming that was *avoided* due to the actual historical drops in these non-CO2 gases.

In [1]:
import fair
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from fair import FAIR
from fair.interface import fill

print(f"Using FaIR Version {fair.__version__}")


/Users/owenhuang/Desktop/Climate/climate_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using FaIR Version 2.2.4


## Step 1: Initialize FaIR and Define the Experiment Setup

FaIR v2.2+ requires defining the time axis, the scenarios, and the species.

In [2]:
# Load properties from FaIR's built-in AR6 species config file
species_csv = os.path.join(os.path.dirname(fair.__file__),
                            'defaults', 'data', 'ar6', 'species_configs_properties.csv')
df_props = pd.read_csv(species_csv, index_col=0)

# Use CO2 FFI + CO2 AFOLU (emission-driven) + CO2 (calculated) — the standard FaIR setup
species_list = ['CO2 FFI', 'CO2 AFOLU', 'CO2', 'CH4', 'N2O', 'CFC-11', 'CFC-12', 'Sulfur']
properties = df_props.loc[species_list].to_dict(orient='index')

# ── First FaIR instance: fetch ssp245 RCMIP data as our 'historical' baseline ──
f_fetch = FAIR()
f_fetch.define_time(1750, 2020, 1)
f_fetch.define_scenarios(['ssp245'])  # ssp245 historical matches observations up to 2020
f_fetch.define_configs(['default'])
f_fetch.define_species(species_list, properties)
f_fetch.allocate()
f_fetch.fill_species_configs()
f_fetch.fill_from_rcmip()            # downloads RCMIP v5.1 files (cached after first run)

print("Downloaded RCMIP base emissions.")


Downloaded RCMIP base emissions.


## Step 2: Load Emissions Data

We use FaIR's built-in `fill_from_rcmip()` to pull historical emissions from the
RCMIP v5.1 protocol (downloaded and cached automatically).  
The **counterfactual** scenario is identical to historical except that CH4, CFC-11,
and CFC-12 are frozen at their 1998 values for all years afterwards.


In [3]:
# ── Main FaIR instance with two scenarios ──────────────────────────────────
f = FAIR()
f.define_time(1750, 2020, 1)
f.define_scenarios(['historical', 'counterfactual_1998_fixed'])
f.define_configs(['default'])
f.define_species(species_list, properties)
f.allocate()
f.fill_species_configs()

# Populate both scenarios with the SSP245 (historical) emissions
for scenario in f.scenarios:
    f.emissions.loc[dict(scenario=scenario)] = (
        f_fetch.emissions.loc[dict(scenario='ssp245')].values
    )

# ── The counterfactual: freeze CH4, CFC-11, and CFC-12 at 1998 levels ────────
year_1998_idx = int(1998 - 1750)   # index in the timepoints array (midpoints)

for specie in ['CH4', 'CFC-11', 'CFC-12']:
    val_1998 = float(
        f.emissions.loc[dict(scenario='counterfactual_1998_fixed', specie=specie)]
        .values[year_1998_idx]
    )
    f.emissions.loc[dict(scenario='counterfactual_1998_fixed', specie=specie)].values[
        year_1998_idx:
    ] = val_1998

print("Emissions loaded. Counterfactual uses frozen 1998 levels for CH4, CFC-11, CFC-12.")


TypeError: only 0-dimensional arrays can be converted to Python scalars

## Step 3: Run the Model and Compare Trends

We run both FaIR scenarios and compute the 1998-2012 linear temperature trend.
The *difference* between the counterfactual and historical trends gives the warming
that was **avoided** by the real-world reductions in non-CO₂ greenhouse gases.


In [ ]:
# ── 1. Run the model ────────────────────────────────────────────────────────
f.run()

# ── 2. Extract surface temperatures for the 1998-2012 window ─────────────────
# timebounds go from 1750 to 2020 (271 values); temperature[i] = temp at timebounds[i]
tb_1998 = int(1998 - 1750)   # timebounds index
tb_2013 = int(2013 - 1750)   # +1 so the 2012 endpoint is included

temp_hist  = f.temperature[tb_1998:tb_2013, 0, 0, 0].values  # layer 0 = surface
temp_cf    = f.temperature[tb_1998:tb_2013, 1, 0, 0].values

# ── 3. Fit linear trends (°C / decade) ───────────────────────────────────────
years = np.arange(1998, 2013)   # 1998 … 2012 inclusive = 15 points
trend_hist = np.polyfit(years, temp_hist, 1)[0] * 10
trend_cf   = np.polyfit(years, temp_cf,   1)[0] * 10
difference = trend_cf - trend_hist

print(f"Historical trend   (1998-2012): {trend_hist:+.4f} °C / decade")
print(f"Counterfactual trend           : {trend_cf:+.4f} °C / decade")
print(f"Warming avoided by CH4/ODS cuts: {difference:+.4f} °C / decade")

# ── 4. Plot ──────────────────────────────────────────────────────────────────
full_years = f.timebounds[int(1900-1750):]  # show from 1900 onwards
temp_hist_full = f.temperature[int(1900-1750):, 0, 0, 0].values
temp_cf_full   = f.temperature[int(1900-1750):, 1, 0, 0].values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: full 1900-2020 trajectories
axes[0].plot(full_years, temp_hist_full, label='Historical', color='steelblue')
axes[0].plot(full_years, temp_cf_full,   label='Counterfactual (frozen 1998)', color='tomato', linestyle='--')
axes[0].axvspan(1998, 2012, alpha=0.15, color='gold', label='1998-2012 window')
axes[0].set_title('Global Mean Temperature Anomaly (1900-2020)')
axes[0].set_ylabel('Temperature change (°C since 1750)')
axes[0].legend()
axes[0].grid(True)

# Right: 1998-2012 close-up with trend lines
axes[1].plot(years, temp_hist, 'o-', color='steelblue', label='Historical')
axes[1].plot(years, temp_cf,   's--', color='tomato', label='Counterfactual')
for col, trend, data, marker in [
    ('steelblue', trend_hist, temp_hist, '-'),
    ('tomato',    trend_cf,   temp_cf,   '--')
]:
    p = np.polyfit(years, data, 1)
    axes[1].plot(years, np.polyval(p, years), color=col, linestyle=marker, alpha=0.5)
axes[1].set_title(f'1998-2012 Warming Slowdown\n'
                  f'Δtrend = {difference:+.4f} °C/decade avoided')
axes[1].set_ylabel('Temperature change (°C since 1750)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()
